# Object Detection using RT-DETRv2

## Setup

In [ ]:
!pip install -q "transformers>=4.44.2" "datasets>=2.21.0" \
               "albumentations>=1.3.0" "torchmetrics>=1.4.0" \
               timm pycocotools scikit-learn tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 57.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from collections import Counter

import torch
from torch.utils.data import Dataset, Subset
from torch.nn.functional import softmax

import albumentations as A
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.model_selection import train_test_split

In [ ]:
from transformers import (
    RTDetrV2ForObjectDetection,
    RTDetrImageProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# GPU check
print("PyTorch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

PyTorch: 2.9.0+cu126
Using device: cuda
GPU name: NVIDIA A100-SXM4-80GB
GPU memory: 85.167243264 GB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Define the base directory as a direct string path for simplicity in a Linux-like environment
DATA_ROOT = os.path.join("/content", "drive", "MyDrive", "Deep_Learning_Project", "Training_and_annotations")

# Define specific paths relative to the DATA_ROOT using os.path.join for robustness
TRAIN_IMAGES_DIR = os.path.join(DATA_ROOT, "training")
TEST_IMAGES_DIR  = os.path.join(DATA_ROOT, "testing")
TRAIN_ANN_PATH   = os.path.join(DATA_ROOT, "train.json")
TEST_ANN_PATH    = os.path.join(DATA_ROOT, "test.json")

print("Train images dir:", TRAIN_IMAGES_DIR, "->", os.path.exists(TRAIN_IMAGES_DIR))
print("Test  images dir:", TEST_IMAGES_DIR,  "->", os.path.exists(TEST_IMAGES_DIR))
print("train.json:", TRAIN_ANN_PATH, "->", os.path.exists(TRAIN_ANN_PATH))
print("test.json: ", TEST_ANN_PATH,  "->", os.path.exists(TEST_ANN_PATH))

print("Sample training files:", os.listdir(TRAIN_IMAGES_DIR)[:5])
print("Sample testing files :", os.listdir(TEST_IMAGES_DIR)[:5])

Mounted at /content/drive
Train images dir: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/training -> True
Test  images dir: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/testing -> True
train.json: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/train.json -> True
test.json:  /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/test.json -> True
Sample training files: ['xray_21607.png', 'xray_21781.png', 'xray_21630.png', 'xray_21778.png', 'xray_21837.png']
Sample testing files : ['xray_hidden01432.png', 'xray_hidden01514.png', 'xray_hidden01422.png', 'xray_hidden01605.png', 'xray_hidden01445.png']


In [ ]:
with open(TRAIN_ANN_PATH, "r") as f:
    train_coco = json.load(f)

categories = train_coco["categories"]
print(f"\nNumber of categories: {len(categories)}")
print("Categories:", categories)

# Contiguous IDs 0..C-1 for the model
id2label = {i: cat["name"] for i, cat in enumerate(categories)}
label2id = {v: k for k, v in id2label.items()}

# Original COCO-style category_id -> contiguous ID
coco_id_to_contiguous = {cat["id"]: i for i, cat in enumerate(categories)}

print("\nid2label:", id2label)
print("coco_id_to_contiguous:", coco_id_to_contiguous)

# Analyze class distribution
class_counts = Counter()
for ann in train_coco["annotations"]:
    contiguous_id = coco_id_to_contiguous[ann["category_id"]]
    class_counts[contiguous_id] += 1

print("\nClass distribution:")
for class_id, count in sorted(class_counts.items()):
    print(f"  {id2label[class_id]}: {count} instances")


Number of categories: 12
Categories: [{'id': 1, 'name': 'Baton'}, {'id': 2, 'name': 'Pliers'}, {'id': 3, 'name': 'Hammer'}, {'id': 4, 'name': 'Powerbank'}, {'id': 5, 'name': 'Scissors'}, {'id': 6, 'name': 'Wrench'}, {'id': 7, 'name': 'Gun'}, {'id': 8, 'name': 'Bullet'}, {'id': 9, 'name': 'Sprayer'}, {'id': 10, 'name': 'HandCuffs'}, {'id': 11, 'name': 'Knife'}, {'id': 12, 'name': 'Lighter'}]

id2label: {0: 'Baton', 1: 'Pliers', 2: 'Hammer', 3: 'Powerbank', 4: 'Scissors', 5: 'Wrench', 6: 'Gun', 7: 'Bullet', 8: 'Sprayer', 9: 'HandCuffs', 10: 'Knife', 11: 'Lighter'}
coco_id_to_contiguous: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9, 11: 10, 12: 11}

Class distribution:
  Baton: 1513 instances
  Pliers: 2000 instances
  Hammer: 2000 instances
  Powerbank: 2000 instances
  Scissors: 2000 instances
  Wrench: 2000 instances
  Gun: 2000 instances
  Bullet: 1837 instances
  Sprayer: 2000 instances
  HandCuffs: 2000 instances
  Knife: 2000 instances
  Lighter: 2000 instances


In [ ]:
IMAGE_SIZE = 640  # Can increase to 800 if GPU memory allows

# Less aggressive augmentation for X-ray images
train_transform = A.Compose(
    [
        A.LongestMaxSize(IMAGE_SIZE),
        A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=0, value=(0, 0, 0)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=5, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
        A.CLAHE(clip_limit=2.0, p=0.2),  # Good for X-ray contrast
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["category"],
        min_visibility=0.3,  # Filter out heavily cropped boxes
        min_area=25,  # Filter out tiny boxes
    ),
)

/tmp/ipython-input-3478421726.py:7: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=0, value=(0, 0, 0)),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-3478421726.py:13: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),


In [ ]:
val_transform = A.Compose(
    [
        A.LongestMaxSize(IMAGE_SIZE),
        A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=0, value=(0, 0, 0)),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["category"]),
)

/tmp/ipython-input-2922711177.py:4: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=0, value=(0, 0, 0)),


In [ ]:
def convert_voc_to_coco(bbox):
    """Pascal VOC [xmin, ymin, xmax, ymax] -> COCO [x, y, w, h]."""
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    return [xmin, ymin, width, height]

def formatted_anns(image_id, categories, areas, bboxes_coco):
    """Convert bboxes + categories to a list of COCO-style annotations."""
    annotations = []
    for cat_id, area, box in zip(categories, areas, bboxes_coco):
        annotations.append(
            {
                "image_id": image_id,
                "category_id": int(cat_id),
                "iscrowd": 0,
                "area": float(area),
                "bbox": list(box),
            }
        )
    return annotations

In [ ]:
def denormalize_cxcywh_to_xyxy(boxes, width, height):
	"""
	boxes: [..., 4] in normalized (cx, cy, w, h)
	width, height: image size in pixels
	"""
	boxes = boxes.clone()
	# scale to pixels
	boxes[:, 0] *= width   # cx
	boxes[:, 1] *= height  # cy
	boxes[:, 2] *= width   # w
	boxes[:, 3] *= height  # h

	cx = boxes[:, 0]
	cy = boxes[:, 1]
	w  = boxes[:, 2]
	h  = boxes[:, 3]

	x1 = cx - 0.5 * w
	y1 = cy - 0.5 * h
	x2 = cx + 0.5 * w
	y2 = cy + 0.5 * h

	return torch.stack([x1, y1, x2, y2], dim=-1)

In [ ]:
class PIDrayRTDetrDataset(Dataset):
    def __init__(self, ann_path, image_dir, image_processor,
                 coco_id_to_contiguous, transform=None):
        super().__init__()
        with open(ann_path, "r") as f:
            coco = json.load(f)

        self.image_dir = image_dir
        self.image_processor = image_processor
        self.transform = transform
        self.coco_id_to_contiguous = coco_id_to_contiguous

        self.images = coco["images"]
        self.annotations = coco["annotations"]

        # Map image_id -> list of annotations
        self.imgid_to_anns = {img["id"]: [] for img in self.images}
        for ann in self.annotations:
            self.imgid_to_anns[ann["image_id"]].append(ann)

        print(f"Loaded {len(self.images)} images from {ann_path}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        image_id = img_info["id"]
        file_name = img_info["file_name"]
        width, height = img_info["width"], img_info["height"]

        img_path = os.path.join(self.image_dir, file_name)
        image = Image.open(img_path).convert("RGB")

        anns = self.imgid_to_anns.get(image_id, [])

        # Build VOC boxes + contiguous class IDs
        bboxes_voc = []
        categories = []
        areas = []

        for ann in anns:
            x, y, w, h = ann["bbox"]  # COCO [x, y, w, h]
            if w <= 1 or h <= 1 or ann.get("iscrowd", 0) == 1:
                continue

            xmin, ymin = x, y
            xmax, ymax = x + w, y + h

            xmin = max(0, min(xmin, width - 1))
            ymin = max(0, min(ymin, height - 1))
            xmax = max(0, min(xmax, width - 1))
            ymax = max(0, min(ymax, height - 1))
            if xmax <= xmin or ymax <= ymin:
                continue

            bboxes_voc.append([xmin, ymin, xmax, ymax])
            contiguous_id = self.coco_id_to_contiguous[ann["category_id"]]
            categories.append(contiguous_id)
            areas.append((xmax - xmin) * (ymax - ymin))

        # Convert to numpy RGB for Albumentations
        image_np = np.array(image)

        if self.transform is not None:
            if len(bboxes_voc) > 0:
                try:
                    out = self.transform(
                        image=image_np,
                        bboxes=bboxes_voc,
                        category=categories,
                    )
                    image_np = out["image"]
                    bboxes_voc = out["bboxes"]
                    categories = out["category"]
                    areas = []
                    for box in bboxes_voc:
                        xmin, ymin, xmax, ymax = box
                        areas.append(max(0.0, (xmax - xmin) * (ymax - ymin)))
                except Exception as e:
                    # If transform fails, use original
                    print(f"Transform failed for image {file_name}: {e}")
                    pass
            else:
                out = self.transform(image=image_np, bboxes=[], category=[])
                image_np = out["image"]

        # Back to PIL RGB for image processor
        image = Image.fromarray(image_np)

        # VOC -> COCO
        bboxes_coco = [convert_voc_to_coco(b) for b in bboxes_voc]
        anns_coco = formatted_anns(image_id, categories, areas, bboxes_coco)

        target = {
            "image_id": image_id,
            "annotations": anns_coco,
        }

        encoding = self.image_processor(
            images=image,
            annotations=target,
            return_tensors="pt",
        )

        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]

        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [ ]:
checkpoint = "PekingU/rtdetr_v2_r50vd"  # Upgraded to R50 for better performance

image_processor = RTDetrImageProcessor.from_pretrained(checkpoint)
num_labels = len(id2label)

model = RTDetrV2ForObjectDetection.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(device)

print(f"\nModel loaded: {checkpoint}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Some weights of RTDetrV2ForObjectDetection were not initialized from the model checkpoint at PekingU/rtdetr_v2_r50vd and are newly initialized because the shapes did not match:
- model.decoder.class_embed.0.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([12]) in the model instantiated
- model.decoder.class_embed.0.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([12, 256]) in the model instantiated
- model.decoder.class_embed.1.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([12]) in the model instantiated
- model.decoder.class_embed.1.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([12, 256]) in the model instantiated
- model.decoder.class_embed.2.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([12]) in the model instantiated
- model.decoder.class_embed.2.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([12, 256]) in the model instantiated
- model


Model loaded: PekingU/rtdetr_v2_r50vd
Number of parameters: 42.75M


In [ ]:
# Split training data into train and validation
all_train_images = train_coco["images"]
train_indices, val_indices = train_test_split(
    range(len(all_train_images)),
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print(f"\nDataset splits:")
print(f"  Training: {len(train_indices)} images")
print(f"  Validation: {len(val_indices)} images")

# Create full training dataset
full_train_dataset = PIDrayRTDetrDataset(
    ann_path=TRAIN_ANN_PATH,
    image_dir=TRAIN_IMAGES_DIR,
    image_processor=image_processor,
    coco_id_to_contiguous=coco_id_to_contiguous,
    transform=train_transform,
)

# Create validation dataset (no augmentation)
val_dataset_base = PIDrayRTDetrDataset(
    ann_path=TRAIN_ANN_PATH,
    image_dir=TRAIN_IMAGES_DIR,
    image_processor=image_processor,
    coco_id_to_contiguous=coco_id_to_contiguous,
    transform=val_transform,
)

# Subset datasets
train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(val_dataset_base, val_indices)

# Test dataset
test_dataset = PIDrayRTDetrDataset(
    ann_path=TEST_ANN_PATH,
    image_dir=TEST_IMAGES_DIR,
    image_processor=image_processor,
    coco_id_to_contiguous=coco_id_to_contiguous,
    transform=val_transform,
)

print(f"  Test: {len(test_dataset)} images")

# Quick sanity check
sample = full_train_dataset[0]
print("\nSample check:")
print("  pixel_values shape:", sample["pixel_values"].shape)
print("  labels keys:", sample["labels"].keys())
print("  labels['boxes'] shape:", sample["labels"]["boxes"].shape)
print("  labels['class_labels']:", sample["labels"]["class_labels"])


Dataset splits:
  Training: 15521 images
  Validation: 2740 images
Loaded 18261 images from /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/train.json
Loaded 18261 images from /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/train.json
Loaded 2922 images from /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/test.json
  Test: 2922 images

Sample check:
  pixel_values shape: torch.Size([3, 640, 640])
  labels keys: KeysView({'size': tensor([640, 640]), 'image_id': tensor([0]), 'class_labels': tensor([0]), 'boxes': tensor([[0.6761, 0.5497, 0.1304, 0.4117]]), 'area': tensor([21980.4941]), 'iscrowd': tensor([0]), 'orig_size': tensor([640, 640])})
  labels['boxes'] shape: torch.Size([1, 4])
  labels['class_labels']: tensor([0])


In [ ]:
import numpy as np

def get_unique_labels(dataset, id2label):
    unique_labels = set()
    for i in range(len(dataset)):
        # Access the actual dataset if it's a Subset
        current_dataset = dataset.dataset if isinstance(dataset, Subset) else dataset
        idx_in_full_dataset = dataset.indices[i] if isinstance(dataset, Subset) else i

        # Get original annotations for this image from the full dataset
        # Need to re-access annotations as the __getitem__ returns processed data
        img_info = current_dataset.images[idx_in_full_dataset]
        image_id = img_info["id"]

        anns = current_dataset.imgid_to_anns.get(image_id, [])
        for ann in anns:
            contiguous_id = current_dataset.coco_id_to_contiguous[ann["category_id"]]
            unique_labels.add(contiguous_id)
    return unique_labels

# Get expected labels
all_expected_label_ids = set(id2label.keys())

# Check training dataset
print("Checking Training Dataset Labels...")
labels_present_in_train = get_unique_labels(train_dataset, id2label)

print(f"  Number of unique labels found in training dataset: {len(labels_present_in_train)}")
print(f"  Labels found (IDs): {sorted(list(labels_present_in_train))}")
print(f"  Labels found (Names): {[id2label[lid] for lid in sorted(list(labels_present_in_train))]}")

if labels_present_in_train == all_expected_label_ids:
    print("  -> All 12 labels are covered in the training dataset!")
else:
    missing_train_labels = all_expected_label_ids - labels_present_in_train
    print(f"  -> WARNING: Missing labels in training dataset: {[id2label[lid] for lid in sorted(list(missing_train_labels))]}")

print("\nChecking Validation Dataset Labels...")
# Check validation dataset
labels_present_in_val = get_unique_labels(val_dataset, id2label)

print(f"  Number of unique labels found in validation dataset: {len(labels_present_in_val)}")
print(f"  Labels found (IDs): {sorted(list(labels_present_in_val))}")
print(f"  Labels found (Names): {[id2label[lid] for lid in sorted(list(labels_present_in_val))]}")

if labels_present_in_val == all_expected_label_ids:
    print("  -> All 12 labels are covered in the validation dataset!")
else:
    missing_val_labels = all_expected_label_ids - labels_present_in_val
    print(f"  -> WARNING: Missing labels in validation dataset: {[id2label[lid] for lid in sorted(list(missing_val_labels))]}")

print("\nChecking Test Dataset Labels...")
# Check test dataset
labels_present_in_test = get_unique_labels(test_dataset, id2label)

print(f"  Number of unique labels found in test dataset: {len(labels_present_in_test)}")
print(f"  Labels found (IDs): {sorted(list(labels_present_in_test))}")
print(f"  Labels found (Names): {[id2label[lid] for lid in sorted(list(labels_present_in_test))]}")

if labels_present_in_test == all_expected_label_ids:
    print("  -> All 12 labels are covered in the test dataset!")
else:
    missing_test_labels = all_expected_label_ids - labels_present_in_test
    print(f"  -> WARNING: Missing labels in test dataset: {[id2label[lid] for lid in sorted(list(missing_test_labels))]}")

Checking Training Dataset Labels...
  Number of unique labels found in training dataset: 12
  Labels found (IDs): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Labels found (Names): ['Baton', 'Pliers', 'Hammer', 'Powerbank', 'Scissors', 'Wrench', 'Gun', 'Bullet', 'Sprayer', 'HandCuffs', 'Knife', 'Lighter']
  -> All 12 labels are covered in the training dataset!

Checking Validation Dataset Labels...
  Number of unique labels found in validation dataset: 12
  Labels found (IDs): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Labels found (Names): ['Baton', 'Pliers', 'Hammer', 'Powerbank', 'Scissors', 'Wrench', 'Gun', 'Bullet', 'Sprayer', 'HandCuffs', 'Knife', 'Lighter']
  -> All 12 labels are covered in the validation dataset!

Checking Test Dataset Labels...
  Number of unique labels found in test dataset: 12
  Labels found (IDs): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  Labels found (Names): ['Baton', 'Pliers', 'Hammer', 'Powerbank', 'Scissors', 'Wrench', 'Gun', 'Bullet', 'Sprayer', 'HandCuffs',

In [ ]:
def collate_fn(batch):
    pixel_values = [item["pixel_values"] for item in batch]
    enc = image_processor.pad(pixel_values, return_tensors="pt")
    labels = [item["labels"] for item in batch]
    batch_out = {
        "pixel_values": enc["pixel_values"],
        "pixel_mask": enc["pixel_mask"],
        "labels": labels,
    }
    return batch_out

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers.modeling_outputs import ModelOutput
import torch

metric = MeanAveragePrecision(box_format="xyxy", class_metrics=True)

@torch.no_grad()
def compute_metrics(eval_pred):
	predictions = eval_pred.predictions
	target_batches = eval_pred.label_ids

	preds = []
	targets = []

	metric.reset()

	# --- loop over batches ---
	for batch_outputs, batch_targets in zip(predictions, target_batches):
		# target_sizes: (batch_size, 2) [h, w]
		target_sizes = torch.tensor(
			[t["orig_size"] for t in batch_targets],
			dtype=torch.long,
		)

		# build GT targets for this batch
		for t in batch_targets:
			h, w = map(float, t["orig_size"])  # (h, w)
			gt_boxes_norm = torch.tensor(t["boxes"], dtype=torch.float32)
			gt_labels = torch.tensor(t["class_labels"], dtype=torch.int64)

			gt_boxes_abs_xyxy = denormalize_cxcywh_to_xyxy(gt_boxes_norm, w, h)
			targets.append(
				{
					"boxes": gt_boxes_abs_xyxy,
					"labels": gt_labels,
				}
			)

		# unpack model outputs (loss, logits, pred_boxes, ...)
		batch_logits, batch_boxes = batch_outputs[1], batch_outputs[2]

		output = ModelOutput(
			logits=torch.tensor(batch_logits),
			pred_boxes=torch.tensor(batch_boxes),
		)

		post_processed = image_processor.post_process_object_detection(
			output,
			threshold=0.05,  # try >0 once things work
			target_sizes=target_sizes,
		)
		preds.extend(post_processed)

	# --- compute mAP ---
	m = metric(preds, targets)

	classes = m.pop("classes")
	map_per_class = m.pop("map_per_class")
	mar_100_per_class = m.pop("mar_100_per_class")

	for class_id, c_map, c_mar in zip(classes, map_per_class, mar_100_per_class):
		class_name = id2label[int(class_id)]
		m[f"map_{class_name}"] = c_map
		m[f"mar_100_{class_name}"] = c_mar

	return {k: float(v) for k, v in m.items()}


## Training

In [ ]:
import os
from transformers import (
	TrainingArguments,
	Trainer,
	RTDetrImageProcessor, # Changed from AutoImageProcessor
	AutoModelForObjectDetection
)

DATA_ROOT = DATA_ROOT  # assuming you already set this earlier

# This must be the SAME output_dir you used in the first training run
output_dir = os.path.join(DATA_ROOT, "rt_detr_v2_pidray_finetuned")

# Find the latest checkpoint
def get_last_checkpoint(output_dir: str):
	checkpoints = [
		d for d in os.listdir(output_dir)
		if d.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, d))
	]
	if not checkpoints:
		raise ValueError(f"No checkpoints found in {output_dir}.")
	# sort by the numeric part after "checkpoint-"
	checkpoints.sort(key=lambda x: int(x.split("-")[1]))
	last_ckpt = os.path.join(output_dir, checkpoints[-1])
	return last_ckpt

last_checkpoint = get_last_checkpoint(output_dir)
print(f"Resuming training from checkpoint: {last_checkpoint}")

# Load model & processor from checkpoint
model = AutoModelForObjectDetection.from_pretrained(last_checkpoint)
# Fix: Load image processor from the original model checkpoint, as it's not changed during fine-tuning
image_processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_v2_r50vd")

# New TrainingArguments
training_args = TrainingArguments(
	output_dir=output_dir,

	# Batch size & accumulation (same as before)
	per_device_train_batch_size=16,
	per_device_eval_batch_size=16,
	gradient_accumulation_steps=1,

	num_train_epochs=25,

	# Optimizer / LR schedule (same as before)
	learning_rate=5e-5,
	weight_decay=1e-4,
	lr_scheduler_type="cosine",
	warmup_ratio=0.05,
	max_grad_norm=1.0,
	optim="adamw_torch",

	# Logging / eval / saving
	logging_dir=os.path.join(output_dir, "logs"),
	logging_steps=25,
	eval_strategy="epoch",  # same as your eval_strategy
	save_strategy="epoch",
	save_total_limit=3,
	load_best_model_at_end=True,
	metric_for_best_model="map",
	greater_is_better=True,

	# Mixed precision
	fp16=True,
	fp16_full_eval=False,

	# Dataloader
	remove_unused_columns=False,
	dataloader_num_workers=4,
	dataloader_pin_memory=True,

	# Reporting
	report_to="tensorboard",

	# Reproducibility
	seed=42,
	eval_do_concat_batches=False,
)

print("\nNew training configuration (resume):")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Total epochs (new): {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim}")
print(f"  Resuming from checkpoint: {last_checkpoint}")

# Create Trainer
trainer = Trainer(
	model=model,
	args=training_args,
	data_collator=collate_fn,
	train_dataset=train_dataset,
	eval_dataset=val_dataset,
	processing_class=image_processor,
	compute_metrics=compute_metrics,
)

Resuming training from checkpoint: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned/checkpoint-420

New training configuration (resume):
  Output dir: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned
  Total epochs (new): 25
  Learning rate: 5e-05
  Optimizer: OptimizerNames.ADAMW_TORCH
  Resuming from checkpoint: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned/checkpoint-420


In [ ]:
trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['class_embed.0.weight', 'class_embed.0.bias', 'class_embed.1.weight', 'class_embed.1.bias', 'class_embed.2.weight', 'class_embed.2.bias', 'class_embed.3.weight', 'class_embed.3.bias', 'class_embed.4.weight', 'class_embed.4.bias', 'class_embed.5.weight', 'class_embed.5.bias', 'bbox_embed.0.layers.0.weight', 'bbox_embed.0.layers.0.bias', 'bbox_embed.0.layers.1.weight', 'bbox_embed.0.layers.1.bias', 'bbox_embed.0.layers.2.weight', 'bbox_embed.0.layers.2.bias', 'bbox_embed.1.layers.0.weight', 'bbox_embed.1.layers.0.bias', 'bbox_embed.1.layers.1.weight', 'bbox_embed.1.layers.1.bias', 'bbox_embed.1.layers.2.weight', 'bbox_embed.1.layers.2.bias', 'bbox_embed.2.layers.0.weight', 'bbox_embed.2.layers.0.bias', 'bbox_embed.2.layers.1.weight', 'bbox_embed.2.layers.1.bias', 'bbox_embed.2.layers.2.weight', 'bbox_embed.2.layers.2.bias', 'bbox_embed.3.layers.0.weight', 'bbox_embed.3.layers.0.bias', 'bbox_embed.3.layers.1.weight', 'bbox_embed.3.l

Epoch,Training Loss,Validation Loss,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map Baton,Mar 100 Baton,Map Pliers,Mar 100 Pliers,Map Hammer,Mar 100 Hammer,Map Powerbank,Mar 100 Powerbank,Map Scissors,Mar 100 Scissors,Map Wrench,Mar 100 Wrench,Map Gun,Mar 100 Gun,Map Bullet,Mar 100 Bullet,Map Sprayer,Mar 100 Sprayer,Map Handcuffs,Mar 100 Handcuffs,Map Knife,Mar 100 Knife,Map Lighter,Mar 100 Lighter
2,13.426300,7.262898,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000565,0.000000,0.001063,0.000548,0.000000,0.003077,0.000000,0.000000,0.000000,0.003289,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000412,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,13.081900,6.834016,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000164,0.000000,0.000000,0.000329,0.000000,0.000000,0.000000,0.000000,0.000000,0.001974,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,12.285700,6.430523,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000411,0.000000,0.000636,0.000439,0.000000,0.000000,0.000000,0.000000,0.000000,0.003947,0.000000,0.000000,0.000000,0.000000,0.000000,0.000980,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,11.632400,6.207617,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001165,0.000000,0.001042,0.001535,0.000000,0.000000,0.000000,0.000000,0.000001,0.013158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000823,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,11.448100,6.053099,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000421,0.000000,0.000683,0.000439,0.000000,0.000769,0.000000,0.000000,0.000000,0.002632,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001646,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,11.160300,6.030707,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000055,0.000862,0.000000,0.000868,0.001096,0.000000,0.000000,0.000000,0.000000,0.000001,0.009868,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000472,0.000000,0.000000
8,10.784000,5.895011,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000110,0.000213,0.000000,0.000333,0.000219,0.000000,0.000000,0.000000,0.000000,0.000000,0.001316,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001235,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,10.815700,5.810338,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000502,0.000000,0.000410,0.000601,0.000000,0.000000,0.000000,0.000000,0.000000,0.004605,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001415,0.000000,0.000000
10,10.517700,5.772931,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000055,0.000164,0.000000,0.000273,0.000110,0.000000,0.000000,0.000000,0.000000,0.000000,0.001974,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11,10.061700,5.776532,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000439,0.000000,0.000410,0.000548,0.000000,0.000000,0.000000,0.000000,0.000001,0.005263,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
/usr/local/l

TrainOutput(global_step=6075, training_loss=9.31382413322543, metrics={'train_runtime': 9929.6179, 'train_samples_per_second': 39.078, 'train_steps_per_second': 0.612, 'total_flos': 2.11670064954291e+20, 'train_loss': 9.31382413322543, 'epoch': 25.0})

In [ ]:
import os
from transformers import (
	TrainingArguments,
	Trainer,
	RTDetrImageProcessor, # Changed from AutoImageProcessor
	AutoModelForObjectDetection
)

DATA_ROOT = DATA_ROOT  # assuming you already set this earlier

# This must be the SAME output_dir you used in the first training run
output_dir = os.path.join(DATA_ROOT, "rt_detr_v2_pidray_finetuned")

# Find the latest checkpoint
def get_last_checkpoint(output_dir: str):
	checkpoints = [
		d for d in os.listdir(output_dir)
		if d.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, d))
	]
	if not checkpoints:
		raise ValueError(f"No checkpoints found in {output_dir}.")
	# sort by the numeric part after "checkpoint-"
	checkpoints.sort(key=lambda x: int(x.split("-")[1]))
	last_ckpt = os.path.join(output_dir, checkpoints[-1])
	return last_ckpt

last_checkpoint = get_last_checkpoint(output_dir)
print(f"Resuming training from checkpoint: {last_checkpoint}")

# Load model & processor from checkpoint
model = AutoModelForObjectDetection.from_pretrained(last_checkpoint)
# Fix: Load image processor from the original model checkpoint, as it's not changed during fine-tuning
image_processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_v2_r50vd")

# New TrainingArguments
training_args = TrainingArguments(
	output_dir=output_dir,

	# Batch size & accumulation (same as before)
	per_device_train_batch_size=64,
	per_device_eval_batch_size=64,
	gradient_accumulation_steps=1,

	num_train_epochs=25,

	# Optimizer / LR schedule (same as before)
	learning_rate=5e-5,
	weight_decay=1e-4,
	lr_scheduler_type="cosine",
	warmup_ratio=0.05,
	max_grad_norm=1.0,
	optim="adamw_torch",

	# Logging / eval / saving
	logging_dir=os.path.join(output_dir, "logs"),
	logging_steps=25,
	eval_strategy="epoch",  # same as your eval_strategy
	save_strategy="epoch",
	save_total_limit=3,
	load_best_model_at_end=True,
	metric_for_best_model="map",
	greater_is_better=True,

	# Mixed precision
	fp16=True,
	fp16_full_eval=False,

	# Dataloader
	remove_unused_columns=False,
	dataloader_num_workers=4,
	dataloader_pin_memory=True,

	# Reporting
	report_to="tensorboard",

	# Reproducibility
	seed=42,
	eval_do_concat_batches=False,
)

print("\nNew training configuration (resume):")
print(f"  Output dir: {training_args.output_dir}")
print(f"  Total epochs (new): {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim}")
print(f"  Resuming from checkpoint: {last_checkpoint}")

# Create Trainer
trainer = Trainer(
	model=model,
	args=training_args,
	data_collator=collate_fn,
	train_dataset=train_dataset,
	eval_dataset=val_dataset,
	processing_class=image_processor,
	compute_metrics=compute_metrics,
)

Resuming training from checkpoint: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned/checkpoint-6075

New training configuration (resume):
  Output dir: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned
  Total epochs (new): 25
  Learning rate: 5e-05
  Optimizer: OptimizerNames.ADAMW_TORCH
  Resuming from checkpoint: /content/drive/MyDrive/Deep_Learning_Project/Training_and_annotations/rt_detr_v2_pidray_finetuned/checkpoint-6075


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Map,Map 50,Map 75,Map Small,Map Medium,Map Large,Mar 1,Mar 10,Mar 100,Mar Small,Mar Medium,Mar Large,Map Baton,Mar 100 Baton,Map Pliers,Mar 100 Pliers,Map Hammer,Mar 100 Hammer,Map Powerbank,Mar 100 Powerbank,Map Scissors,Mar 100 Scissors,Map Wrench,Mar 100 Wrench,Map Gun,Mar 100 Gun,Map Bullet,Mar 100 Bullet,Map Sprayer,Mar 100 Sprayer,Map Handcuffs,Mar 100 Handcuffs,Map Knife,Mar 100 Knife,Map Lighter,Mar 100 Lighter
1,8.940700,5.724428,0.756411,0.885872,0.810511,0.218461,0.555242,0.793495,0.827537,0.858882,0.858882,0.276923,0.707166,0.880926,0.750780,0.861712,0.860055,0.893478,0.888140,0.956401,0.631811,0.809119,0.613123,0.725676,0.925027,0.946181,0.841963,0.864769,0.669016,0.847407,0.742542,0.868210,0.927441,0.940864,0.628237,0.806790,0.598801,0.785981
2,9.423500,5.771537,0.777052,0.905551,0.835000,0.208014,0.580703,0.804065,0.828423,0.857281,0.857281,0.261538,0.723721,0.877609,0.791039,0.860360,0.855873,0.886594,0.908332,0.950865,0.626826,0.813982,0.622584,0.702027,0.918526,0.943750,0.850540,0.872598,0.734960,0.842222,0.766784,0.890432,0.918179,0.932226,0.689437,0.821914,0.641545,0.770405
3,9.084400,5.800464,0.746306,0.875259,0.800341,0.224835,0.569938,0.776429,0.816148,0.847247,0.847247,0.292308,0.721406,0.866309,0.780274,0.877477,0.858000,0.891304,0.880866,0.940138,0.613958,0.796353,0.595417,0.687838,0.920425,0.944792,0.854481,0.880427,0.678899,0.844074,0.780993,0.878395,0.918943,0.931229,0.491339,0.708333,0.582078,0.786604
4,9.209500,5.821963,0.739643,0.863504,0.792028,0.177806,0.537283,0.777982,0.815082,0.844763,0.844763,0.326923,0.663343,0.864051,0.810440,0.881081,0.855450,0.883333,0.896212,0.949481,0.606233,0.799696,0.601184,0.697297,0.925486,0.943056,0.855253,0.880427,0.644797,0.846667,0.699690,0.893519,0.925264,0.938538,0.451569,0.659259,0.604142,0.764798
5,8.973100,5.846766,0.730173,0.854463,0.781249,0.215780,0.534398,0.754224,0.802973,0.828017,0.828017,0.269231,0.658696,0.848812,0.743910,0.863514,0.856352,0.884783,0.859563,0.944291,0.508983,0.684195,0.605145,0.703716,0.907703,0.923958,0.833884,0.859786,0.673251,0.830370,0.654872,0.787654,0.917011,0.931894,0.620608,0.775309,0.580797,0.746729
6,8.964000,5.862407,0.742532,0.865188,0.799190,0.237499,0.521215,0.770561,0.807720,0.831874,0.831874,0.276923,0.623233,0.853442,0.782893,0.860811,0.865537,0.892754,0.881504,0.940484,0.587741,0.723100,0.615465,0.703716,0.921911,0.943403,0.847448,0.869039,0.679955,0.832593,0.781492,0.870679,0.914241,0.929900,0.472975,0.653086,0.559228,0.762928
7,8.817900,5.892766,0.739187,0.855981,0.794299,0.243101,0.527467,0.768308,0.790253,0.812927,0.812927,0.280769,0.605655,0.832321,0.784794,0.860811,0.861498,0.887319,0.824834,0.879931,0.603425,0.720669,0.630246,0.685473,0.911008,0.934028,0.843658,0.866904,0.661945,0.819259,0.791982,0.855864,0.913851,0.927907,0.524187,0.651543,0.518819,0.665421
8,8.912300,5.816274,0.716616,0.838822,0.769871,0.246352,0.517838,0.742671,0.793162,0.816955,0.816955,0.326923,0.630343,0.833520,0.758751,0.880631,0.845993,0.873551,0.863763,0.957785,0.536717,0.727660,0.631384,0.730068,0.903090,0.922569,0.846377,0.871174,0.640693,0.814444,0.744119,0.881790,0.916624,0.930897,0.420460,0.520988,0.491420,0.691900
9,8.992000,5.799206,0.742171,0.871951,0.791064,0.235892,0.583528,0.762679,0.801996,0.829033,0.829033,0.303846,0.711947,0.842976,0.769904,0.872523,0.860702,0.887319,0.881675,0.940138,0.610285,0.747416,0.621924,0.703716,0.905653,0.929861,0.853938,0.875445,0.644975,0.834444,0.769488,0.866667,0.920485,0.935880,0.487560,0.583333,0.579457,0.771651
10,8.588700,5.812063,0.692691,0.811303,0.741277,0.241071,0.547485,0.704543,0.753901,0.778207,0.778207,0.284615,0.668432,0.781358,0.772633,0.873874,0.866423,0.894565,0.841298,0.917647,0.539513,0.676596,0.598555,0.704054,0.906594,0.930208,0.858130,0.877580,0.508411,0.668148,0.693900,0.795988,0.928586,0.939867,0.343150,0.417901,0.455098,0.642056


There were missing keys in the checkpoint model loaded: ['class_embed.0.weight', 'class_embed.0.bias', 'class_embed.1.weight', 'class_embed.1.bias', 'class_embed.2.weight', 'class_embed.2.bias', 'class_embed.3.weight', 'class_embed.3.bias', 'class_embed.4.weight', 'class_embed.4.bias', 'class_embed.5.weight', 'class_embed.5.bias', 'bbox_embed.0.layers.0.weight', 'bbox_embed.0.layers.0.bias', 'bbox_embed.0.layers.1.weight', 'bbox_embed.0.layers.1.bias', 'bbox_embed.0.layers.2.weight', 'bbox_embed.0.layers.2.bias', 'bbox_embed.1.layers.0.weight', 'bbox_embed.1.layers.0.bias', 'bbox_embed.1.layers.1.weight', 'bbox_embed.1.layers.1.bias', 'bbox_embed.1.layers.2.weight', 'bbox_embed.1.layers.2.bias', 'bbox_embed.2.layers.0.weight', 'bbox_embed.2.layers.0.bias', 'bbox_embed.2.layers.1.weight', 'bbox_embed.2.layers.1.bias', 'bbox_embed.2.layers.2.weight', 'bbox_embed.2.layers.2.bias', 'bbox_embed.3.layers.0.weight', 'bbox_embed.3.layers.0.bias', 'bbox_embed.3.layers.1.weight', 'bbox_embed.3.l

TrainOutput(global_step=6075, training_loss=8.418612939654064, metrics={'train_runtime': 8934.1854, 'train_samples_per_second': 43.431, 'train_steps_per_second': 0.68, 'total_flos': 1.2229566131109888e+20, 'train_loss': 8.418612939654064, 'epoch': 25.0})

## Evaluation

In [ ]:
print("Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)

print("\nTest Set Results:")
print(f"  mAP: {test_results.get('eval_map', 0):.4f}")
print(f"  mAP@50: {test_results.get('eval_map_50', 0):.4f}")
print(f"  mAP@75: {test_results.get('eval_map_75', 0):.4f}")

# Print per-class results
print("\nPer-class mAP:")
for class_name in id2label.values():
    key = f"eval_map_{class_name}"
    if key in test_results:
        print(f"  {class_name}: {test_results[key]:.4f}")

Evaluating on test set...


/tmp/ipython-input-2289655038.py:20: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  target_sizes = torch.tensor(



Test Set Results:
  mAP: 0.4766
  mAP@50: 0.5801
  mAP@75: 0.5143

Per-class mAP:
  Baton: 0.5926
  Pliers: 0.8066
  Hammer: 0.5429
  Powerbank: 0.2260
  Scissors: 0.6346
  Wrench: 0.7206
  Gun: 0.3163
  Bullet: 0.3072
  Sprayer: 0.3658
  HandCuffs: 0.8813
  Knife: 0.1689
  Lighter: 0.1563


In [ ]:
model.eval()
model.to(device)

def visualize_prediction(img_path, score_threshold=0.5):
    """Visualize predictions on a single image and return the annotated image."""
    image = Image.open(img_path).convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([(image.height, image.width)], device=device)
    results = image_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=score_threshold
    )[0]

    draw = ImageDraw.Draw(image)]

    # print(f"\nDetections for {os.path.basename(img_path)}:")
    for score, label_id, box in zip(results["scores"], results["labels"], results["boxes"]):
        score = float(score)
        label_id = int(label_id)
        x1, y1, x2, y2 = [float(x) for x in box]

        label_name = id2label[label_id]
        # print(f"  {label_name}: {score:.3f} at [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

        # Draw box
        draw.rectangle((x1, y1, x2, y2), outline="red", width=3)

        # Draw label
        text = f"{label_name} {score:.2f}"
        draw.text((x1, max(0, y1 - 15)), text, fill="red")

    return image

# Test on multiple images
print("\n" + "="*60)
print("Running inference on test samples...")
print("="*60)

test_images = os.listdir(TEST_IMAGES_DIR)[:12]  # Test on 12 images

fig, axes = plt.subplots(4, 3, figsize=(20, 20)) # Changed to 4 rows, 3 columns
axes = axes.flatten()

for i, img_name in enumerate(test_images):
    img_path = os.path.join(TEST_IMAGES_DIR, img_name)
    annotated_image = visualize_prediction(img_path, score_threshold=0.5)

    axes[i].imshow(annotated_image)
    axes[i].set_title(f"Predictions (th=0.5)\n{os.path.basename(img_path)}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
save_path = os.path.join(output_dir, "predictions_grid.png")
plt.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved prediction grid to: {save_path}")
plt.show()

print("\n" + "="*60)
print("All done! Check your Google Drive for saved models and results.")
print("="*60)

Output hidden; open in https://colab.research.google.com to view.